# Osteosarcoma Data Visualization Project
![split-bone-revealing-osteosarcoma-tumor-600nw-2521894175](split-bone-revealing-osteosarcoma-tumor-600nw-2521894175.webp) 

## Acquiring and Loading Data
### Importing Libraries and Notebook Setup

In [ ]:
%pip install numpy pandas matplotlib seaborn dash plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import dash
from dash import Dash, dcc, html
from dash.dependencies import Input, Output


### Loading Data

In [ ]:
#Load the clinical patient data
df = pd.read_csv("data_clinical_patient.txt", sep="\t", comment="#", skiprows=4)

### Basic Data Exploration

In [ ]:
#View structure
print(df.columns)
df.head()


### Handling Missing Values

In [ ]:
###Removing Incomplete Rows
df = df.dropna(subset=["SEX", "RACE", "OS_STATUS", "AGE"])


### QUESTION 1 : Mortality Rate by Sex and Race
### Is there a racial disparity in mortality rates between male and female osteosarcoma patients?


> This analysis explores how mortality rates differ between **male and female osteosarcoma patients** across various **racial groups**

---

### Steps

1. **Selected Relevant Columns**:  
   - `SEX`: Patient’s sex (male or female)  
   - `RACE`: Reported racial background of the patient  
   - `VITAL_STATUS`: Whether the patient was alive or dead  

2. **Data Cleaning**:
   - Removed any rows with missing values in these columns.
   - Converted all text to uppercase to ensure consistent grouping.

---

### Mortality Rate by Sex and Race

Grouped the data by `RACE` and `SEX`, then used `value_counts(normalize=True)` to calculate the proportion of patients who  **died** in each group.

- Filtered to include only `"DEAD"` values, so the chart shows **true mortality rates**.
- **X-axis**: Race categories  
- **Y-axis**: Mortality rate (as a percentage)  
- **Bars**: Grouped by sex (male vs female)

> This visualization highlights whether certain racial groups and sexes have a higher likelihood of mortality from osteosarcoma.


In [ ]:
#Select relevant columns
df_q1 = df[['SEX', 'RACE', 'VITAL_STATUS']].dropna()

#Clean text format
df_q1['SEX'] = df_q1['SEX'].str.upper()
df_q1['RACE'] = df_q1['RACE'].str.upper()
df_q1['VITAL_STATUS'] = df_q1['VITAL_STATUS'].str.upper()

#Calculate mortality rate
mortality = (
    df_q1
    .groupby(['RACE', 'SEX'])['VITAL_STATUS']
    .value_counts(normalize=True)
    .rename('rate')
    .reset_index()
)

#Keep only "DEAD" rows for mortality
mortality = mortality[mortality['VITAL_STATUS'] == 'DEAD']

#Plot
fig = px.bar(
    mortality,
    x='RACE',
    y='rate',
    color='SEX',
    barmode='group',
    title='Mortality Rate by Sex and Race',
    labels={'rate': 'Mortality Rate', 'RACE': 'Race'}
)

fig.update_layout(
    yaxis_tickformat=".0%",
    title_font_size=20,
    legend_title_text='Sex',
    xaxis_title='Race',
    yaxis_title='Mortality Rate'
)

fig.show()

### QUESTION 2: Metastasis at Diagnosis based on Sex and Its Impact on Mortality
### Are males more likely than females to present with metastasis at diagnosis and does this affect survival?

> This analysis explores whether there are differences in the presence of metastasis at the time of diagnosis between male and female osteosarcoma patients and how metastasis affects mortality outcomes.

---
### Steps

1. **Selected Relevant Columns**:  
   - `SEX`: Patient’s sex (male or female)
   - `METASTASIS_AT_DIAGNOSIS`: Whether metastases were detected at diagnosis  
   - `VITAL_STATUS`: Whether the patient was alive or dead

2. **Data Cleaning**:
   - Removed any rows with missing values in these columns.
   - Converted text fields to uppercase to avoid grouping inconsistencies.

---

### Firstly: Metastasis at Diagnosis Based on Sex

grouped patients by `SEX` and `METASTASIS_AT_DIAGNOSIS` to count how many in each group had or did not have metastasis at diagnosis.

- **X-axis**: Metastasis status ("YES" or "NO")
- **Y-axis**: Number of patients
- **Bars**: Grouped by sex (male or female)

> This visualization helps identify which sex is more likely to present with metastasis at diagnosis.

---

### Secondly: Mortality Rate Based on Metastasis Status

analyzed how the presence of metastasis at diagnosis correlates with mortality:

- Used `value_counts(normalize=True)` to calculate the **percentage of patients who died** in each metastasis group.
- Filtered to include only the `"DEAD"` status to reflect true mortality rate.

- **X-axis**: Metastasis at diagnosis (YES/NO)
- **Y-axis**: Mortality rate (%)

> This chart reveals whether patients with metastasis at diagnosis had significantly higher mortality.

---

> These visualizations work together to reveal both demographic patterns and clinical outcomes related to metastasis in osteosarcoma.


In [ ]:
# Select relevant columns
df_q2 = df[['SEX', 'METASTASIS_AT_DIAGNOSIS', 'VITAL_STATUS']].dropna()

# Clean text
df_q2['SEX'] = df_q2['SEX'].str.upper()
df_q2['METASTASIS_AT_DIAGNOSIS'] = df_q2['METASTASIS_AT_DIAGNOSIS'].str.upper()
df_q2['VITAL_STATUS'] = df_q2['VITAL_STATUS'].str.upper()

#Frequency of metastasis at diagnosis based on sex

meta_freq = (
    df_q2
    .groupby(['SEX', 'METASTASIS_AT_DIAGNOSIS'])
    .size()
    .reset_index(name='count')
)

fig_a = px.bar(
    meta_freq,
    x='METASTASIS_AT_DIAGNOSIS',
    y='count',
    color='SEX',
    barmode='group',
    title='Metastasis at Diagnosis based on Sex',
    labels={'count': 'Number of Patients', 'METASTASIS_AT_DIAGNOSIS': 'Metastasis at Diagnosis'}
)
fig_a.update_layout(title_font_size=20)
fig_a.show()


# Mortality rate based on metastasis presence

meta_mortality = (
    df_q2
    .groupby(['METASTASIS_AT_DIAGNOSIS'])['VITAL_STATUS']
    .value_counts(normalize=True)
    .rename('rate')
    .reset_index()
)

# Keep only the pateints who died
meta_mortality = meta_mortality[meta_mortality['VITAL_STATUS'] == 'DEAD']

fig_b = px.bar(
    meta_mortality,
    x='METASTASIS_AT_DIAGNOSIS',
    y='rate',
    title='Mortality Rate based on Metastasis at Diagnosis',
    labels={'rate': 'Mortality Rate', 'METASTASIS_AT_DIAGNOSIS': 'Metastasis at Diagnosis'}
)
fig_b.update_layout(
    yaxis_tickformat=".0%",
    title_font_size=20,
    xaxis_title='Metastasis Status',
    yaxis_title='Mortality Rate'
)
fig_b.show()

### QUESTION 3: Tumor Site Distribution and Associated Mortality Rates

> Do tumor locations differ significantly between male and female patients and are some sites linked to worse outcomes?


> This analysis explores wether **tumor location (primary site)** varies between males and females, and how tumor site may relate to **mortality outcomes**.

1. **Selected Columns**:  
   - `SEX`: Patient’s sex (male or female)
   - `PRIMARY_SITE_PATIENT`: Anatomical site of the tumor  
   - `VITAL_STATUS`: Whether the patient was alive or dead

2. **Data Cleaning**:
   - Dropped rows with missing values
   - Standardized all text to uppercase for consistency in grouping

---

### Firstly: Tumor Site Distribution Based on Sex

counted how many patients had tumors in each site, grouped by sex.

- **X-axis**: Tumor site  
- **Y-axis**: Number of patients  
- **Bars**: Colored by sex (male vs female)

> This visualization reveals whether certain tumor sites are more common in males or females.

---

### Secondly: Overall Mortality Rate Based on Tumor Site

Here, we examined the **percentage of dead patients only** for each tumor site (regardless of sex).

- Used `value_counts(normalize=True)` to calculate the proportion of deaths at each site
- Filtered only the `"DEAD"` values

- **X-axis**: Tumor site  
- **Y-axis**: Mortality rate (in percentage)  

**Note**: The percentages shown in this chart represent the proportion of dead patients **relative to the total number of patients (dead and alive)** at each tumor site.

> This visualization is useful for identifying tumor locations associated with higher risk.

---

### Thirdly: Mortality Rate based on Tumor Site and Sex

Finally, analyzed mortality rates **based on both tumor site and sex**.

- Used `value_counts(normalize=True)` to calculate the proportion of deaths at each site
- Filtered by both `SEX` and `PRIMARY_SITE_PATIENT`

- **X-axis**: Tumor site  
- **Y-axis**: Mortality rate (%)  
- **Bars**: Colored by sex

**Note**: The percentages shown in this chart represent the proportion of dead patients **relative to the total number of patients (dead and alive)** at each tumor site.

>  This visualization is useful for identifying whether male or female patients have worse outcomes depending on tumor location.

---

> The three visualization provide a comprehensive view of how tumor site and sex intersect with survival in osteosarcoma patients.

In [ ]:
# Select relevant columns
df_q3 = df[['SEX', 'PRIMARY_SITE_PATIENT', 'VITAL_STATUS']].dropna()

#Clean text format
df_q3['SEX'] = df_q3['SEX'].str.upper()
df_q3['PRIMARY_SITE_PATIENT'] = df_q3['PRIMARY_SITE_PATIENT'].str.upper()
df_q3['VITAL_STATUS'] = df_q3['VITAL_STATUS'].str.upper()


# Tumor site distribution based on sex

tumor_freq = (
    df_q3
    .groupby(['PRIMARY_SITE_PATIENT', 'SEX'])
    .size()
    .reset_index(name='count')
)

fig_a = px.bar(
    tumor_freq,
    x='PRIMARY_SITE_PATIENT',
    y='count',
    color='SEX',
    barmode='group',
    title='Tumor Site Distribution based on Sex',
    labels={'PRIMARY_SITE_PATIENT': 'Primary Tumor Site', 'count': 'Number of Patients'}
)
fig_a.update_layout(title_font_size=20, xaxis_tickangle=45)
fig_a.show()


#  Overall mortality rate based on tumor site

mortality_overall = (
    df_q3
    .groupby('PRIMARY_SITE_PATIENT')['VITAL_STATUS']
    .value_counts(normalize=True)
    .rename('rate')
    .reset_index()
)
mortality_overall = mortality_overall[mortality_overall['VITAL_STATUS'] == 'DEAD']

fig_b = px.bar(
    mortality_overall,
    x='PRIMARY_SITE_PATIENT',
    y='rate',
    title='Mortality Rate based on Tumor Site (All Patients)',
    labels={'PRIMARY_SITE_PATIENT': 'Primary Tumor Site', 'rate': 'Mortality Rate'}
)
fig_b.update_layout(
    yaxis_tickformat=".0%",
    title_font_size=20,
    xaxis_tickangle=45
)
fig_b.show()

# Mortality rate based on tumor site and sex

mortality_by_sex = (
    df_q3
    .groupby(['PRIMARY_SITE_PATIENT', 'SEX'])['VITAL_STATUS']
    .value_counts(normalize=True)
    .rename('rate')
    .reset_index()
)
mortality_by_sex = mortality_by_sex[mortality_by_sex['VITAL_STATUS'] == 'DEAD']

fig_c = px.bar(
    mortality_by_sex,
    x='PRIMARY_SITE_PATIENT',
    y='rate',
    color='SEX',
    barmode='group',
    title='Mortality Rate based on Tumor Site and Sex',
    labels={'PRIMARY_SITE_PATIENT': 'Primary Tumor Site', 'rate': 'Mortality Rate'}
)
fig_c.update_layout(
    yaxis_tickformat=".0%",
    title_font_size=20,
    xaxis_tickangle=45
)
fig_c.show()

### QUESTION 4: Diagnosis Age Distribution between Males and Females
Do males and females get diagnosed at different ages and does age at diagnosis affect outcome?

> This analysis explores whether there are differences in **age at diagnosis** between male and female osteosarcoma patients, and how age is distributed within each sex group.

---

### Steps

1. **Selected Relevant Columns**:  
   - `SEX`: Patient’s sex (male or female)  
   - `AGE`: Age at diagnosis of osteosarcoma

2. **Data Cleaning**:
   - Removed any rows with missing values in these columns.
   - Converted text fields to uppercase to ensure consistency.
   - Age groups were created by mapping:
     - `AGE > (or) = 23`: exact integer (e.g., 12, 15, 23)
     - `AGE > 23`: grouped into a single category `'>24'`

---

### Firstly: Age Distribution based on Sex (Grouped Bar Chart)

Patients were grouped by `SEX` and `AGE_GROUP_COMBINED` to compare the number of diagnoses at each age level.

- **X-axis**: Age group 
- **Y-axis**: Number of patients  
- **Bars**: Grouped by sex (male or female)

> This visualization shows whether male or female are diagnosed earlier or more frequently at a specific ages.

---

### Secondly: Age Distribution based on Sex (Pie Charts)

To better understand how age is distributed **within** each sex group, the **percentage of each age group** relative to all patients of the same sex was calculated
 Created two pie charts:
  - One for **female** patients
  - One for **male** patients
- **Labels**: Age groups  
- **Values**: Percentage of patients in each group

> These pie charts reveal internal age distribution patterns for each sex and highlight the most common ages at diagnosis.

---

> These visualizations show both **absolute counts** and **relative proportions** of osteosarcoma diagnoses by age and sex.


In [ ]:
## Bar Chart

# Select relevant columns

df_q4 = df[['SEX', 'AGE']].dropna()
df_q4 = df_q4[df_q4['AGE'] <= 40]
df_q4['SEX'] = df_q4['SEX'].str.upper()

# Categorize age: 1–23 stay as-is.
# >23 becomes '>24'

def age_group(age):
    return str(int(age)) if age <= 23 else '>24'

df_q4['AGE_GROUP_COMBINED'] = df_q4['AGE'].apply(age_group)
df_q4_grouped = df_q4.copy()


# Sort Age Groups 
def age_sort_key(value):
    try:
        return int(value)
    except ValueError:
        return 999  

sorted_ages = sorted(df_q4_grouped['AGE_GROUP_COMBINED'].unique(), key=age_sort_key)
df_q4_grouped['AGE_GROUP_COMBINED'] = pd.Categorical(df_q4_grouped['AGE_GROUP_COMBINED'], categories=sorted_ages, ordered=True)

# Group and count again for visualization
age_dist_sorted = df_q4_grouped.groupby(['SEX', 'AGE_GROUP_COMBINED']).size().reset_index(name='count')

# Plot 

fig_sorted = px.bar(
    age_dist_sorted,
    x='AGE_GROUP_COMBINED',
    y='count',
    color='SEX',
    barmode='group',
    title='Diagnosis Age Distribution between Males and Females',
    labels={'AGE_GROUP_COMBINED': 'Age Group', 'count': 'Number of Patients'}
)
fig_sorted.update_layout(title_font_size=20, xaxis_tickangle=45)
fig_sorted.show()




In [ ]:

# Select relevant columns
df_q4 = df[['SEX', 'AGE']].dropna()
df_q4 = df_q4[df_q4['AGE'] <= 40]
df_q4['SEX'] = df_q4['SEX'].str.upper()

#Define Age Groups
def age_group(age):
    return str(int(age)) if age <= 23 else '>24'

df_q4['AGE_GROUP_COMBINED'] = df_q4['AGE'].apply(age_group)

# Calculate percentage within each sex
age_dist_pie = (
    df_q4
    .groupby(['SEX', 'AGE_GROUP_COMBINED'])
    .size()
    .reset_index(name='count')
)

age_dist_pie['percentage'] = age_dist_pie.groupby('SEX')['count'].transform(lambda x: x / x.sum() * 100)

# Sort Age Groups
age_dist_pie['AGE_GROUP_COMBINED'] = age_dist_pie['AGE_GROUP_COMBINED'].astype(str)
age_dist_pie['AGE_SORTED'] = age_dist_pie['AGE_GROUP_COMBINED'].apply(lambda x: int(x) if x.isdigit() else 999)
age_dist_pie = age_dist_pie.sort_values(by=['SEX', 'AGE_SORTED'])

# Plot
fig_female = px.pie(
    age_dist_pie[age_dist_pie['SEX'] == 'FEMALE'],
    values='percentage',
    names='AGE_GROUP_COMBINED',
    title='Diagnosis Age Distribution (FEMALE)',
    labels={'AGE_GROUP_COMBINED': 'Age Group'}
)

fig_male = px.pie(
    age_dist_pie[age_dist_pie['SEX'] == 'MALE'],
    values='percentage',
    names='AGE_GROUP_COMBINED',
    title='Diagnosis Age Distribution (MALE)',
    labels={'AGE_GROUP_COMBINED': 'Age Group'}
)

fig_female.show()
fig_male.show()




### Question 5: ICD-10 Classification between Male and Female
How Do ICD-10 Diagnostic Classifications Differ Between Male and Female Osteosarcoma Patients?


> This dashboard explores whether there are differences in teh distibution of **ICD-10 classifications** between male and female osteosarcoma patients using an interactive web interface.

---

### Steps

1. **Selected Relevant Columns**:  
   - `SEX`: Patient’s sex (male or female)  
   - `ICD_10`: ICD-10 diagnostic code assigned to each patient

2. **Data Cleaning**:
   - Removed any rows with missing values in these columns using `.dropna()`
   - Standardized the text fields by converting all values to uppercase to avoid grouping mismatches

---

### Dashboard Functionality

A dropdown menu allows users to choose between:
- `"All"`: Shows ICD-10 classifications for both sexes
- `"Male"`: Displays only data for male patients
- `"Female"`: Displays only data for female patients

A grouped **bar chart** is updated based on the selected option.

- **X-axis**: ICD-10 classification codes  
- **Y-axis**: Number of patients per ICD-10 code  
- **Bars**: Colored by sex (if viewing both sexes)

> This dashboard provides an interactive way to explore diagnostic code patterns across male and female osteosarcoma patients.

---

> This visualization highlights how diagnostic classifications are distributed across both males and females and offers a flexible tool to filter and compare patients using ICD-10 patterns.


In [ ]:

# Clean and filter
df_icd = df[['SEX', 'ICD_10']].dropna()
df_icd['SEX'] = df_icd['SEX'].str.upper()
df_icd['ICD_10'] = df_icd['ICD_10'].str.upper()

# ------------------------------------
# Start Dash App
# ------------------------------------
app = dash.Dash(__name__)
app.title = "ICD-10 Classification by Sex"

app.layout = html.Div([
    html.H2("ICD-10 Classification Distribution based on Sex", style={"textAlign": "center"}),

    html.Label("Select Sex:", style={"fontWeight": "bold", "marginTop": "20px"}),
    dcc.Dropdown(
        id='sex-dropdown',
        options=[
            {'label': 'All', 'value': 'ALL'},
            {'label': 'Male', 'value': 'MALE'},
            {'label': 'Female', 'value': 'FEMALE'}
        ],
        value='ALL',
        style={"width": "300px"}
    ),

    dcc.Graph(id='icd-graph')
])


# ------------------------------------
# Callback for updating the graph
# ------------------------------------
@app.callback(
    Output('icd-graph', 'figure'),
    [Input('sex-dropdown', 'value')]
)
def update_icd_graph(selected_sex):
    if selected_sex == 'ALL':
        filtered = df_icd.copy()
    else:
        filtered = df_icd[df_icd['SEX'] == selected_sex]

    icd_counts = (
        filtered
        .groupby(['ICD_10', 'SEX'])
        .size()
        .reset_index(name='count')
    )

    fig = px.bar(
        icd_counts,
        x='ICD_10',
        y='count',
        color='SEX' if selected_sex == 'ALL' else None,
        title='ICD-10 Classification Distribution between male and female',
        labels={'ICD_10': 'ICD-10 Code', 'count': 'Number of Patients'},
        barmode='group'
    )

    fig.update_layout(xaxis_tickangle=45, title_font_size=20)
    return fig


# ------------------------------------
# Run the App
# ------------------------------------
if __name__ == '__main__':
    app.run (debug=True)
